In [17]:
import boto3
import io
import re
import requests
from datasets import load_dataset
import chardet
import os
from PIL import Image
import numpy as np

In [2]:
s3 = boto3.client(
    "s3",
    endpoint_url="http://127.0.0.1:9000", # MinIO API endpoint
   #endpoint_url="https://tirelessly-unslouching-delaine.ngrok-free.dev", # MinIO API endpoint (ngrok)
    aws_access_key_id="minioadmin", # User name
    aws_secret_access_key="minioadmin", # Password
)

In [3]:
# List existing buckets
buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]

# Function that given a name, creates a bucket
def createBucket(name, list_buckets):
    if name in list_buckets:
        print(f"Bucket '{name}' already exists!")
    else:
        s3.create_bucket(Bucket=name)
        print(f"Created bucket: {name}")

# Create a bucket named test_zone
createBucket("test-zone", buckets)
# Create sub-buckets inside test_zone.
s3.put_object(Bucket="test-zone", Key="image-data/") # Landing zone for raw game descriptions.
s3.put_object(Bucket="test-zone", Key="text-data/") # Landing zone for downloaded game assets.
s3.put_object(Bucket="test-zone", Key="test-data/") # Final repository for aligned, normalized multimodal pairs.

Bucket 'test-zone' already exists!


{'ResponseMetadata': {'RequestId': '18857ED6D11AB615',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'etag': '"d41d8cd98f00b204e9800998ecf8427e"',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-checksum-crc32': 'AAAAAA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18857ED6D11AB615',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '2107',
   'x-ratelimit-remaining': '2107',
   'x-xss-protection': '1; mode=block',
   'date': 'Sun, 28 Dec 2025 21:38:09 GMT'},
  'RetryAttempts': 0},
 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
 'ChecksumCRC32': 'AAAAAA==',
 'ChecksumType': 'FULL_OBJECT'}

We load game metadata from two sources: **Steam** and **RAWG**. To evaluate our models on unseen data, we extract the **last 200 entries** from each dataset. This fixed-size hold-out set ensures we have a consistent benchmark for comparing different model versions.

In [4]:
# We are going to use two datasets, one from: https://huggingface.co/datasets/FronkonGames/steam-games-dataset (123 MB)
ds1_raw = load_dataset("FronkonGames/steam-games-dataset")
# The other one from: https://huggingface.co/datasets/atalaydenknalbant/rawg-games-dataset (998 MB)
ds2_raw = load_dataset("atalaydenknalbant/rawg-games-dataset")
# We are going to use the last 200 obs as test set because dataset don't contain test split
ds1 = ds1_raw['train'][-200:]
ds2 = ds2_raw['train'][-200:]

# Print the number of rows of each subdataset
print(f"The subdataset 1 contains {len(ds1['About the game'])} rows")
print(f"The subdataset 2 contains {len(ds2['description'])} rows")

The subdataset 1 contains 200 rows
The subdataset 2 contains 200 rows


In this stage, we move raw data from the dataframes into object storage:
* **Text Ingestion**: Converts game descriptions into `.txt` files encoded in `UTF-8`.
* **Image Ingestion**: Uses stream-downloading to fetch headers and background images.
* **Robustness**: The pipeline includes exception handling to skip broken links (404 errors) or request timeouts, ensuring the notebook continues execution without interruption.

**Uploading Texts**

In [5]:
# This function uploads each game description from strings to the given bucket_name,
# saving them as separate text files (text_1.txt, text_2.txt, …).
def upload_strings_separately(bucket_name, client, strings, path="raw-data/", prefix="text"):
    for i, s in enumerate(strings, start=1):
        if not s: # skip empty strings or None
            continue
        object_name = f"{path}{prefix}_{i}.txt" # temporal-landing/text_1.txt, temporal-landing/text_2.txt ...
        client.put_object(
            Bucket=bucket_name,
            Key=object_name,
            Body=io.BytesIO(s.encode("utf-8")),
            ContentType="text/plain"
        )

        print(f"Uploaded: {object_name}")


In [6]:
# Uploading text files (combining both datasets)
upload_strings_separately("test-zone", s3, strings =
                          ds1['About the game'] +
                          ds2['description'],
                          path = "text-data/")

Uploaded: text-data/text_1.txt
Uploaded: text-data/text_2.txt
Uploaded: text-data/text_3.txt
Uploaded: text-data/text_4.txt
Uploaded: text-data/text_5.txt
Uploaded: text-data/text_6.txt
Uploaded: text-data/text_7.txt
Uploaded: text-data/text_8.txt
Uploaded: text-data/text_9.txt
Uploaded: text-data/text_11.txt
Uploaded: text-data/text_12.txt
Uploaded: text-data/text_13.txt
Uploaded: text-data/text_14.txt
Uploaded: text-data/text_15.txt
Uploaded: text-data/text_16.txt
Uploaded: text-data/text_17.txt
Uploaded: text-data/text_19.txt
Uploaded: text-data/text_21.txt
Uploaded: text-data/text_22.txt
Uploaded: text-data/text_23.txt
Uploaded: text-data/text_25.txt
Uploaded: text-data/text_27.txt
Uploaded: text-data/text_28.txt
Uploaded: text-data/text_29.txt
Uploaded: text-data/text_30.txt
Uploaded: text-data/text_31.txt
Uploaded: text-data/text_32.txt
Uploaded: text-data/text_34.txt
Uploaded: text-data/text_35.txt
Uploaded: text-data/text_37.txt
Uploaded: text-data/text_38.txt
Uploaded: text-da

**Uploading Images**


In [7]:
# This function downloads URL in links as a stream and uploads it directly to the given bucket,
# saving the files with names like image_1.jpg, image_2.png, etc., while preserving their extensions.
def upload_media_from_links(bucket_name, client, links, path="temporal-landing/", prefix="image"):
    for i, url in enumerate(links, start=1):
        if not url:
            continue

        try:
            # Stream download to avoid loading full file in memory
            with requests.get(url, stream=True, timeout=120) as r:

                if r.status_code == 404:
                    print(f"Skipped (404 Not Found): {url}") # 404 -> skip
                    continue

                r.raise_for_status() # check for HTTP errors
                ext = url.split('.')[-1].split('?')[0] # get file extension
                object_name = f"{path}{prefix}_{i}.{ext}"
                # This streams the request directly to MinIO without creating a full BytesIO object
                client.upload_fileobj(
                    Fileobj=r.raw,
                    Bucket=bucket_name,
                    Key=object_name,
                    ExtraArgs={"ContentType": f"{prefix}/{ext}"}
                )

                print(f"Uploaded: {object_name}")
        except requests.exceptions.Timeout: #
            print(f"Skipped (Timeout): {url}") # Timeout -> skip
        except requests.exceptions.HTTPError as e:
            print(f"HTTP error for {url}: {e}") # Other HTTP errors -> log + skip
        except Exception as e:
            print(f"Failed {url}: {e}") # Other exceptions -> log + skip

In [8]:
# Uploading image files (combining both datasets)
upload_media_from_links("test-zone", s3, links =
                         ds1['Header image'] + ds2['background_image'], # + ds2['background_image_additional'] + ds1ss + ds2ss,
                         path="image-data/") # If this process is taking too long, we can just skip the screeshots

Uploaded: image-data/image_1.jpg
Uploaded: image-data/image_2.jpg
Uploaded: image-data/image_3.jpg
Uploaded: image-data/image_4.jpg
Uploaded: image-data/image_5.jpg
Uploaded: image-data/image_6.jpg
Uploaded: image-data/image_7.jpg
Uploaded: image-data/image_8.jpg
Uploaded: image-data/image_9.jpg
Uploaded: image-data/image_10.jpg
Uploaded: image-data/image_11.jpg
Uploaded: image-data/image_12.jpg
Uploaded: image-data/image_13.jpg
Uploaded: image-data/image_14.jpg
Uploaded: image-data/image_15.jpg
Uploaded: image-data/image_16.jpg
Uploaded: image-data/image_17.jpg
Uploaded: image-data/image_18.jpg
Uploaded: image-data/image_19.jpg
Uploaded: image-data/image_20.jpg
Uploaded: image-data/image_21.jpg
Uploaded: image-data/image_22.jpg
Uploaded: image-data/image_23.jpg
Uploaded: image-data/image_24.jpg
Uploaded: image-data/image_25.jpg
Uploaded: image-data/image_26.jpg
Uploaded: image-data/image_27.jpg
Uploaded: image-data/image_28.jpg
Uploaded: image-data/image_29.jpg
Uploaded: image-data/im

## Data Normalization for Evaluation

To use this dataset for evaluation, we apply minimal normalization.

### Text Data
- Convert all text to **UTF-8 encoding**

### Image Data
- Convert images to **PNG** format
- Normalize pixel values
- Convert images to **RGB** channels

In [9]:
def normalize_text(client, bucket, prefix=""):
    """
    Scan all objects under a given S3 prefix, normalize text encodings to UTF-8,

    client          : obj                   - S3-compatible client (e.g., boto3.client("s3")).
    bucket          : str                   - Target S3/MinIO bucket.
    prefix          : str                   - Optional key prefix (acts like a folder path).

    """

    # Ensure path ends with '/'
    if prefix and not prefix.endswith("/"):
        prefix += "/"
    paginator = client.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix)

    for page in pages:
        for obj in page.get("Contents", []):

            if obj['Size'] == 0:  # Skip the folder itself (if the file size is 0)
                continue

            key = obj["Key"]

            try:
                # Get the file object from S3
                resp = client.get_object(Bucket=bucket, Key=key)
                body = resp["Body"].read()  # Read the file content

                # Use chardet to detect the file encoding
                result = chardet.detect(body)
                current_encoding = result['encoding']
                # Skip if the file is already in UTF-8 encoding
                if current_encoding in ("utf-8", "ascii", None):
                    print(f"Already UTF-8 or ASCII: {key}")
                    continue

                print(f"Converting {key} from {current_encoding} to UTF-8")

                # Decode the content using the detected encoding and re-encode it in UTF-8
                content = body.decode(current_encoding, errors='ignore')  # Ignore characters that can't be decoded

                client.put_object(
                    Bucket=bucket,
                    Key=key,  # Make sure the file key (path) is correct
                    Body=content.encode('utf-8'),
                    ContentType="text/plain"
                )
                print(f"Re-encoded {key} successfully")


            except Exception as e:
                print(f"Failed to process {key}: {e}")

In [10]:
normalize_text(client=s3,bucket="test-zone",prefix="text-data/")

Already UTF-8 or ASCII: text-data/text_1.txt
Already UTF-8 or ASCII: text-data/text_100.txt
Already UTF-8 or ASCII: text-data/text_101.txt
Already UTF-8 or ASCII: text-data/text_102.txt
Already UTF-8 or ASCII: text-data/text_103.txt
Already UTF-8 or ASCII: text-data/text_104.txt
Already UTF-8 or ASCII: text-data/text_105.txt
Already UTF-8 or ASCII: text-data/text_106.txt
Already UTF-8 or ASCII: text-data/text_107.txt
Converting text-data/text_108.txt from Windows-1254 to UTF-8
Re-encoded text-data/text_108.txt successfully
Already UTF-8 or ASCII: text-data/text_109.txt
Converting text-data/text_11.txt from Windows-1252 to UTF-8
Re-encoded text-data/text_11.txt successfully
Already UTF-8 or ASCII: text-data/text_110.txt
Already UTF-8 or ASCII: text-data/text_111.txt
Already UTF-8 or ASCII: text-data/text_112.txt
Already UTF-8 or ASCII: text-data/text_113.txt
Already UTF-8 or ASCII: text-data/text_115.txt
Already UTF-8 or ASCII: text-data/text_116.txt
Already UTF-8 or ASCII: text-data/te

In [11]:
#This function normalize image data, but don't change the content
def normalize_image(bucket, prefix="", target_size=(512, 512)):
    """
    Normalize an image for machine learning purposes without altering its visual content.

    This function performs minimal preprocessing on an input image to prepare it for
    evaluation or model input. The following steps are applied:

    1. Converts the image to RGB format (3 channels) if it is not already in RGB.
    2. Resizes the image to the specified target size using a high-quality resampling filter (LANCZOS).
    3. Normalizes the pixel values to the range [0, 1] for numerical stability in ML models.
    4. Converts the normalized array back to an 8-bit PIL Image for storage or further processing.
    """
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]

            # Skip folders
            if obj["Size"] == 0 and key.endswith("/"):
                continue

            # Skip already PNG images
            if os.path.splitext(key)[1].lower() == ".png":
                continue

            new_key = os.path.splitext(key)[0] + ".png"

            try:
                # Download image from S3
                resp = s3.get_object(Bucket=bucket, Key=key)
                body = resp["Body"].read()

                # Load image with Pillow
                img = Image.open(io.BytesIO(body))

                # Convert to RGB (3 channels)
                if img.mode != "RGB":
                    img = img.convert("RGB")

                # Resize image
                img = img.resize(target_size, Image.LANCZOS)

                # Normalize pixel values to [0, 1]
                img_array = np.array(img).astype(np.float32) / 255.0

                # Convert back to uint8 for saving
                img_out = Image.fromarray((img_array * 255).astype(np.uint8))

                # Save as PNG to buffer
                buf = io.BytesIO()
                img_out.save(buf, format="PNG")
                buf.seek(0)

                # Upload PNG back to S3
                s3.upload_fileobj(
                    buf,
                    Bucket=bucket,
                    Key=new_key,
                    ExtraArgs={"ContentType": "image/png"}
                )

                # Delete original file
                s3.delete_object(Bucket=bucket, Key=key)

                print(f"Replaced: {key} -> {new_key}")

            except Exception as e:
                print(f"Failed to process {key}: {e}")

In [12]:
normalize_image(bucket="test-zone", prefix="image-data/", target_size=(512, 512))

Replaced: image-data/image_1.jpg -> image-data/image_1.png
Replaced: image-data/image_10.jpg -> image-data/image_10.png
Replaced: image-data/image_100.jpg -> image-data/image_100.png
Replaced: image-data/image_101.jpg -> image-data/image_101.png
Replaced: image-data/image_102.jpg -> image-data/image_102.png
Replaced: image-data/image_103.jpg -> image-data/image_103.png
Replaced: image-data/image_104.jpg -> image-data/image_104.png
Replaced: image-data/image_105.jpg -> image-data/image_105.png
Replaced: image-data/image_106.jpg -> image-data/image_106.png
Replaced: image-data/image_107.jpg -> image-data/image_107.png
Replaced: image-data/image_108.jpg -> image-data/image_108.png
Replaced: image-data/image_109.jpg -> image-data/image_109.png
Replaced: image-data/image_11.jpg -> image-data/image_11.png
Replaced: image-data/image_110.jpg -> image-data/image_110.png
Replaced: image-data/image_111.jpg -> image-data/image_111.png
Replaced: image-data/image_112.jpg -> image-data/image_112.png


The following cell handles the final transfer and validation of the preprocessed test set:
1. **Data Transfer**: Moves the processed files to the `test-data` bucket.
2. **Pairing Verification**: Ensures **multimodal integrity** by checking that every text entry has a corresponding image (and vice-versa).
3. **Filtering**: Automatically excludes any unpaired or incomplete samples to ensure a consistent evaluation set.

In [13]:
# This function copies all objects from a subbucket (under a prefix) into another subbucket in same bucket
def move_files(bucket, dest_prefix, src_prefix=""):
    # Create destination bucket if it doesn't exist


    paginator = s3.get_paginator("list_objects_v2") # It returns objects in pages and not all at once.
    for page in paginator.paginate(Bucket=bucket, Prefix=src_prefix):
        for obj in page.get("Contents", []):

            key = obj["Key"]

            if obj['Size'] == 0 and key.endswith("/"): # skip the folder itself
                continue

            # Remove the prefix part from the key
            new_key = dest_prefix + key[len(src_prefix):]

            # Copy object without top-level folder
            copy_source = {"Bucket": bucket, "Key": key}
            s3.copy_object(Bucket=bucket, Key=new_key, CopySource=copy_source)

            print(f"Copied: {key} -> {new_key}")

In [15]:
move_files(bucket = "test-zone", dest_prefix="test-data", src_prefix = "text-data")

Copied: text-data/text_1.txt -> test-data/text_1.txt
Copied: text-data/text_100.txt -> test-data/text_100.txt
Copied: text-data/text_101.txt -> test-data/text_101.txt
Copied: text-data/text_102.txt -> test-data/text_102.txt
Copied: text-data/text_103.txt -> test-data/text_103.txt
Copied: text-data/text_104.txt -> test-data/text_104.txt
Copied: text-data/text_105.txt -> test-data/text_105.txt
Copied: text-data/text_106.txt -> test-data/text_106.txt
Copied: text-data/text_107.txt -> test-data/text_107.txt
Copied: text-data/text_108.txt -> test-data/text_108.txt
Copied: text-data/text_109.txt -> test-data/text_109.txt
Copied: text-data/text_11.txt -> test-data/text_11.txt
Copied: text-data/text_110.txt -> test-data/text_110.txt
Copied: text-data/text_111.txt -> test-data/text_111.txt
Copied: text-data/text_112.txt -> test-data/text_112.txt
Copied: text-data/text_113.txt -> test-data/text_113.txt
Copied: text-data/text_115.txt -> test-data/text_115.txt
Copied: text-data/text_116.txt -> tes

In [16]:
move_files(bucket = "test-zone", dest_prefix="test-data", src_prefix = "image-data")

Copied: image-data/image_1.png -> test-data/image_1.png
Copied: image-data/image_10.png -> test-data/image_10.png
Copied: image-data/image_100.png -> test-data/image_100.png
Copied: image-data/image_101.png -> test-data/image_101.png
Copied: image-data/image_102.png -> test-data/image_102.png
Copied: image-data/image_103.png -> test-data/image_103.png
Copied: image-data/image_104.png -> test-data/image_104.png
Copied: image-data/image_105.png -> test-data/image_105.png
Copied: image-data/image_106.png -> test-data/image_106.png
Copied: image-data/image_107.png -> test-data/image_107.png
Copied: image-data/image_108.png -> test-data/image_108.png
Copied: image-data/image_109.png -> test-data/image_109.png
Copied: image-data/image_11.png -> test-data/image_11.png
Copied: image-data/image_110.png -> test-data/image_110.png
Copied: image-data/image_111.png -> test-data/image_111.png
Copied: image-data/image_112.png -> test-data/image_112.png
Copied: image-data/image_113.png -> test-data/im

In [18]:
def clean_unpaired_files(s3_client, bucket: str, prefix: str = "test-data/"):
    """
    Deletes unpaired image/text files from an S3/MinIO bucket.
    Only files with matching numeric IDs are kept.
    """
    #  List all object keys under the given prefix
    keys = []
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        keys.extend(obj['Key'] for obj in page.get('Contents', []))

    #  Extract numeric IDs and map to full keys
    imgs = {m.group(1): k for k in keys if (m := re.search(r'image_(\d+)\.png$', k))}
    txts = {m.group(1): k for k in keys if (m := re.search(r'text_(\d+)\.txt$', k))}

    #  Identify orphaned keys (no matching pair)
    orphan_keys = [
        k for id_, k in imgs.items() if id_ not in txts
    ] + [
        k for id_, k in txts.items() if id_ not in imgs
    ]

    #  Delete orphaned files in batches (S3 allows up to 1000 per request)
    if orphan_keys:
        print(f"Deleting {len(orphan_keys)} unpaired files...")
        for i in range(0, len(orphan_keys), 1000):
            batch = orphan_keys[i:i+1000]
            s3_client.delete_objects(
                Bucket=bucket,
                Delete={'Objects': [{'Key': k} for k in batch]}
            )
        print("✅ Cleanup completed.")
    else:
        print("✅ No unpaired files found.")

In [19]:
clean_unpaired_files(s3,"test-zone")

Deleting 21 unpaired files...
✅ Cleanup completed.
